In [0]:
!pip install openai

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import numpy as np
from openai import OpenAI
import os

In [0]:
DATABRICKS_TOKEN = dbutils.entry_point.getDbutils().notebook().getContext().apiToken().get()

In [0]:
DATABRICKS_TOKEN

'[REDACTED]'

In [0]:
client = OpenAI(
  api_key=DATABRICKS_TOKEN,
  base_url="https://dbc-933c892e-dff5.cloud.databricks.com/ai-gateway/mlflow/v1"
)

In [0]:
# Food tracking data
food_logs = {
    1: "Greek yogurt with fresh berries, no added sugar.",
    2: "Greek yogurt with berries and two teaspoons of added sugar.",
    3: "Oatmeal prepared with milk, banana and almonds.",
    4: "Oatmeal prepared with milk, banana and honey.",
    5: "Grilled chicken salad with olive oil dressing, no mayonnaise.",
    6: "Grilled chicken salad with mayonnaise dressing.",
    7: "Vegetable soup with no added salt.",
    8: "Vegetable soup with added salt for taste.",
    9: "Boiled eggs with avocado and whole-wheat toast.",
    10: "Pancakes prepared with refined flour, sugar and maple syrup.",
    11: "Fresh fruit bowl containing apple, banana and berries.",
    12: "Fruit smoothie containing banana, berries and added honey.",
    13: "Unsweetened almond milk with chia seeds.",
    14: "Chocolate protein shake with added sugar.",
    15: "Plain cottage cheese with cucumber and tomatoes."
}

In [0]:
query = "Find meals, I avoid added sugar."


# 1. Generate embeddings
documents = list(food_logs.values())

response = client.embeddings.create(
  input=documents,
  model="system.ai.bge-large-en"
)

document_embeddings = np.array([
    item.embedding for item in response.data
])

In [0]:
document_embeddings

array([[-0.05102539,  0.04623413,  0.0035038 , ...,  0.0033989 ,
         0.03192139,  0.0160675 ],
       [-0.03399658,  0.0098877 ,  0.03182983, ...,  0.02536011,
         0.02160645,  0.02322388],
       [-0.03491211,  0.01293945, -0.00017834, ..., -0.03396606,
         0.0423584 ,  0.02980042],
       ...,
       [-0.06005859,  0.04537964,  0.01947021, ...,  0.02151489,
         0.05984497, -0.01531219],
       [ 0.00428772,  0.0149765 ,  0.02183533, ...,  0.00132847,
         0.03381348,  0.00308037],
       [-0.07324219,  0.03329468,  0.00888824, ...,  0.01291656,
         0.04354858, -0.00192261]], shape=(15, 1024))

In [0]:
query_response = client.embeddings.create(
    model="system.ai.bge-large-en",
    input=query
)

query_embedding = np.array(
    query_response.data[0].embedding
)

In [0]:
query_embedding

array([-0.05609131,  0.00801086, -0.03268433, ..., -0.01062012,
        0.01274872,  0.00696945], shape=(1024,))

In [0]:
# 2. Normalize embeddings (L2 normalization)
def normalize(vectors):
    return vectors / np.linalg.norm(
        vectors, axis=-1, keepdims=True
    )

normalized_documents = normalize(document_embeddings)
normalized_query = normalize(query_embedding)

In [0]:
similarity_scores = np.dot(
    normalized_documents,
    normalized_query
)

top_k = 3

ranked_indices = np.argsort(
    similarity_scores
)[::-1][:top_k]

retrieved_documents = []

for index in ranked_indices:
    retrieved_documents.append({
        "id": index + 1,
        "text": documents[index],
        "similarity": float(similarity_scores[index])
    })


# Display vector similarity results
print("Vector Similarity Results:\n")

for item in retrieved_documents:
    print(
        f"ID: {item['id']} | "
        f"Similarity: {item['similarity']:.4f} | "
        f"{item['text']}"
    )


Vector Similarity Results:

ID: 14 | Similarity: 0.6227 | Chocolate protein shake with added sugar.
ID: 1 | Similarity: 0.6097 | Greek yogurt with fresh berries, no added sugar.
ID: 7 | Similarity: 0.5973 | Vegetable soup with no added salt.


In [0]:

top_k = 6

ranked_indices = np.argsort(
    similarity_scores
)[::-1][:top_k]

retrieved_documents = []

for index in ranked_indices:
    retrieved_documents.append({
        "id": index + 1,
        "text": documents[index],
        "similarity": float(similarity_scores[index])
    })

print(retrieved_documents)
print("*"*30)
import json

# LLM endpoint configured in Databricks Model Serving
CHAT_MODEL = "system.ai.meta-llama-3-1-8b-instruct"

# Prepare retrieved documents for reranking
candidate_text = "\n".join(
    f"ID: {item['id']} | Document: {item['text']}"
    for item in retrieved_documents
)

reranking_prompt = f"""
You are a relevance-ranking assistant for a food tracking application.

User Query:
{query}

Retrieved Documents:
{candidate_text}

Instructions:

1. Evaluate each document against the actual intent of the query.
2. Do not rank documents based only on shared words or topics.
3. Pay attention to negation, exclusions and contradictory statements.
4. Assign a relevance score from 1 to 5:
   5 = Directly satisfies the query.
   4 = Relevant and satisfies the query.
   3 = Partially relevant.
   2 = Related but does not satisfy the query.
   1 = Irrelevant or contradicts the query.

Return every document exactly once, sorted by relevance score
in descending order.

Return ONLY valid JSON in this format:

{{
    "ranked_results": [
        {{
            "id": 1,
            "relevance_score": 5,
            "reason": "Provide the reason for such ranking."
        }}
    ]
}}
"""

# Ask the LLM to rerank the retrieved candidates
reranking_response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {
            "role": "user",
            "content": reranking_prompt
        }
    ],
    temperature=0
)

# Parse LLM response
llm_output = json.loads(
    reranking_response.choices[0].message.content
)

# Sort results by relevance score and retrieve top 3
top_3_results = sorted(
    llm_output["ranked_results"],
    key=lambda x: x["relevance_score"],
    reverse=True
)[:3]

# Display final top 3 ranked results
print("\nLLM-Assisted Reranking - Top 3 Results:\n")

for result in top_3_results:
    print(
        f"ID: {result['id']} | "
        f"Relevance: {result['relevance_score']}/5 | "
        f"{food_logs[result['id']]}"
    )
    print(f"Reason: {result['reason']}\n")


[{'id': np.int64(14), 'text': 'Chocolate protein shake with added sugar.', 'similarity': 0.6227340015158747}, {'id': np.int64(1), 'text': 'Greek yogurt with fresh berries, no added sugar.', 'similarity': 0.6097450711042652}, {'id': np.int64(7), 'text': 'Vegetable soup with no added salt.', 'similarity': 0.597279648568537}, {'id': np.int64(2), 'text': 'Greek yogurt with berries and two teaspoons of added sugar.', 'similarity': 0.5858679201478958}, {'id': np.int64(8), 'text': 'Vegetable soup with added salt for taste.', 'similarity': 0.5514342588994903}, {'id': np.int64(13), 'text': 'Unsweetened almond milk with chia seeds.', 'similarity': 0.5510940134508641}]
******************************

LLM-Assisted Reranking - Top 3 Results:

ID: 1 | Relevance: 5/5 | Greek yogurt with fresh berries, no added sugar.
Reason: Document contains 'no added sugar' which directly satisfies the query.

ID: 13 | Relevance: 5/5 | Unsweetened almond milk with chia seeds.
Reason: Document does not contain added